# Day 18 — Categorical variables, scaling and preprocessing pipelines

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Preprocessing pipelines

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-018/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import pandas as pd
import numpy as np

# A small, messy dataset: numbers and categories, with gaps in both
df = pd.DataFrame({
    "study_hours": [3, 5, np.nan, 4, 6, 2],
    "attendance": [75, 90, 65, np.nan, 85, 55],
    "city": ["Hyderabad", "Mumbai", "Delhi", "Hyderabad", "Mumbai", np.nan],
    "grade": ["A", "B", "A", "C", np.nan, "B"],
    "result": ["Pass", "Pass", "Fail", "Pass", "Pass", "Fail"]})
print(df)

# Each column's kind and gaps (df.info() shows the same, in a format that differs between pandas versions)
kind = ["number" if pd.api.types.is_numeric_dtype(df[c]) else "text" for c in df.columns]
print(pd.DataFrame({"kind": kind, "non-null": df.notna().sum(), "missing": df.isnull().sum()}).to_string())

   study_hours  attendance       city grade result
0          3.0        75.0  Hyderabad     A   Pass
1          5.0        90.0     Mumbai     B   Pass
2          NaN        65.0      Delhi     A   Fail
3          4.0         NaN  Hyderabad     C   Pass
4          6.0        85.0     Mumbai   NaN   Pass
5          2.0        55.0        NaN     B   Fail
               kind  non-null  missing
study_hours  number         5        1
attendance   number         5        1
city           text         5        1
grade          text         5        1
result         text         6        0


In [ ]:
# Feature groups: each group gets its own preprocessing; the target is kept apart from the inputs
numeric_features = ["study_hours", "attendance"]
categorical_features = ["city", "grade"]
target = "result"
X = df[numeric_features + categorical_features]
y = df[target]
print("Numerical:  ", numeric_features)
print("Categorical:", categorical_features)
print("X:", X.shape, "| y:", y.shape)

Numerical:   ['study_hours', 'attendance']
Categorical: ['city', 'grade']
X: (6, 4) | y: (6,)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                             ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                 ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numeric_features),
                                  ("categorical", categorical_pipeline, categorical_features)])

# What each step learned from these six students
Xt = preprocessor.fit_transform(X)
num = preprocessor.named_transformers_["numeric"]
cat = preprocessor.named_transformers_["categorical"]
print("Medians that fill gaps:   ", num.named_steps["imputer"].statistics_)
print("Means used for scaling:   ", num.named_steps["scaler"].mean_.round(2))
print("Most frequent (fill gaps):", cat.named_steps["imputer"].statistics_)
print("Output shape:", Xt.shape)
cols = [n.split("__")[1] for n in preprocessor.get_feature_names_out()]
print("Columns:", ", ".join(cols[:2]), "+", len(cols) - 2, "0/1 columns:", ", ".join(c.split("_")[1] for c in cols[2:]))

Medians that fill gaps:    [ 4. 75.]
Means used for scaling:    [ 4.   74.17]
Most frequent (fill gaps): ['Hyderabad' 'A']
Output shape: (6, 8)
Columns: study_hours, attendance + 6 0/1 columns: Delhi, Hyderabad, Mumbai, A, B, C


In [ ]:
# The six students after preprocessing: no gaps, numbers scaled, categories as 0/1 columns
def show(rows):
    """Scaled numbers to 2 decimals, the 0/1 category columns as whole numbers, with short headers."""
    short = ["hours", "attend", "Delhi", "Hyderabad", "Mumbai", "grade_A", "grade_B", "grade_C"]
    out = pd.DataFrame(rows.toarray() if hasattr(rows, "toarray") else rows, columns=short).round(2)
    out[short[2:]] = out[short[2:]].astype(int)
    print(out.to_string())

show(Xt)

   hours  attend  Delhi  Hyderabad  Mumbai  grade_A  grade_B  grade_C
0  -0.77    0.07      0          1       0        1        0        0
1   0.77    1.35      0          0       1        0        1        0
2   0.00   -0.78      1          0       0        1        0        0
3   0.00    0.07      0          1       0        0        0        1
4   1.55    0.93      0          0       1        1        0        0
5  -1.55   -1.64      0          1       0        0        1        0


In [ ]:
# handle_unknown="ignore": a city never seen in training becomes all zeros instead of an error
new_student = pd.DataFrame({"study_hours": [5], "attendance": [80], "city": ["Bengaluru"], "grade": ["B"]})
show(preprocessor.transform(new_student))

strict = ColumnTransformer([("numeric", numeric_pipeline, numeric_features),
                            ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                                      ("encoder", OneHotEncoder())]), categorical_features)])
strict.fit(X)
try:
    strict.transform(new_student)
except ValueError as error:
    print("Without it:", str(error).split(" during")[0])

   hours  attend  Delhi  Hyderabad  Mumbai  grade_A  grade_B  grade_C
0   0.77     0.5      0          0       0        0        1        0
Without it: Found unknown categories ['Bengaluru'] in column 0


In [ ]:
from sklearn.model_selection import train_test_split

# The same preprocessor on our 150 students. Class is the categorical column; "Unknown" means missing
students = pd.read_csv("student_data_150.csv")
students["Class"] = students["Class"].replace("Unknown", np.nan)
features_num = ["StudyHours", "Attendance", "PreviousScore", "Age"]
features_cat = ["Class"]
X_all, y_all = students[features_num + features_cat], students["Passed"]
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.15, random_state=42, stratify=y_all)
print("Missing values per column:", X_all.isnull().sum().to_dict())
print(f"Train {X_train.shape} | test {X_test.shape}")

real = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), features_num),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("encoder", OneHotEncoder(handle_unknown="ignore"))]), features_cat)])
X_train_p = real.fit_transform(X_train)          # learn medians, means, spreads and categories from training only
X_test_p = real.transform(X_test)                # apply them unchanged to the test students
print("Processed:", X_train_p.shape, X_test_p.shape)
print("Columns:  ", [n.split("__")[1] for n in real.get_feature_names_out()])
print("Class filled with:", real.named_transformers_["categorical"].named_steps["imputer"].statistics_[0])

Missing values per column: {'StudyHours': 0, 'Attendance': 0, 'PreviousScore': 0, 'Age': 0, 'Class': 5}
Train (127, 5) | test (23, 5)
Processed: (127, 7) (23, 7)
Columns:   ['StudyHours', 'Attendance', 'PreviousScore', 'Age', 'Class_A', 'Class_B', 'Class_C']
Class filled with: A


In [ ]:
# The leak: fit on all 150 students first and the test students shape the statistics the model is scored with
leaky = ColumnTransformer([("numeric", StandardScaler(), features_num)]).fit(X_all)
safe = ColumnTransformer([("numeric", StandardScaler(), features_num)]).fit(X_train)
compare = pd.DataFrame({"mean (train only)": safe.named_transformers_["numeric"].mean_,
                        "mean (all 150)": leaky.named_transformers_["numeric"].mean_}, index=features_num).round(3)
print(compare.to_string())
print("Test students scaled differently:", int((np.abs(leaky.transform(X_test) - safe.transform(X_test)) > 1e-9).any(axis=1).sum()),
      "of", len(X_test))

               mean (train only)  mean (all 150)
StudyHours                 3.614           3.573
Attendance                79.126          79.487
PreviousScore             62.394          62.660
Age                       11.031          11.020
Test students scaled differently: 23 of 23


## 2. Advanced categorical encoding

In [ ]:
# One-hot with an unseen category: categories are sorted alphabetically (Delhi, Hyderabad, Mumbai)
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)   # older scikit-learn: sparse=False
encoder.fit([["Hyderabad"], ["Mumbai"], ["Delhi"]])
print(encoder.categories_[0])
print(encoder.transform([["Hyderabad"], ["Bengaluru"]]))

['Delhi' 'Hyderabad' 'Mumbai']
[[0. 1. 0.]
 [0. 0. 0.]]


In [ ]:
# Frequency encoding: learn the counts from training data, with a fallback for cities never seen
train_cities = pd.DataFrame({"city": ["Hyderabad", "Mumbai", "Hyderabad", "Delhi", "Hyderabad", "Mumbai"]})
frequency_map = train_cities["city"].value_counts().to_dict()
print(frequency_map)
train_cities["city_frequency"] = train_cities["city"].map(frequency_map)
print(train_cities.to_string(index=False))
new_cities = pd.Series(["Mumbai", "Bengaluru"])
print(new_cities.map(frequency_map).fillna(0).astype(int).tolist(), "<- Bengaluru gets the fallback 0")

{'Hyderabad': 3, 'Mumbai': 2, 'Delhi': 1}
     city  city_frequency
Hyderabad               3
   Mumbai               2
Hyderabad               3
    Delhi               1
Hyderabad               3
   Mumbai               2
[2, 0] <- Bengaluru gets the fallback 0


In [ ]:
from sklearn.preprocessing import TargetEncoder

# Target encoding: each city's purchase rate, and what scikit-learn's TargetEncoder makes of it
buyers = pd.DataFrame({"city": ["Hyderabad", "Hyderabad", "Mumbai", "Mumbai", "Delhi"], "purchase": [1, 1, 0, 1, 0]})
print("Raw purchase rate:", buyers.groupby("city")["purchase"].mean().to_dict())
te = TargetEncoder(target_type="binary", smooth="auto").fit(buyers[["city"]], buyers["purchase"])
print("Overall rate:     ", buyers["purchase"].mean())
print("Smoothed:         ", {c: round(float(v), 2) for c, v in zip(te.categories_[0], te.encodings_[0])})

Raw purchase rate: {'Delhi': 0.0, 'Hyderabad': 1.0, 'Mumbai': 0.5}
Overall rate:      0.6
Smoothed:          {'Delhi': 0.0, 'Hyderabad': 1.0, 'Mumbai': 0.53}


In [ ]:
# Rare categories: learn which are common from training data, group the rest as "Other"
pay = pd.DataFrame({"payment_method": ["UPI", "UPI", "UPI", "Card", "Card", "Wallet", "Other"]})
counts = pay["payment_method"].value_counts()
common_categories = counts[counts >= 2].index
pay["payment_method_grouped"] = pay["payment_method"].where(pay["payment_method"].isin(common_categories), "Other")
print(counts.to_dict())
print(pay.to_string(index=False))

{'UPI': 3, 'Card': 2, 'Wallet': 1, 'Other': 1}
payment_method payment_method_grouped
           UPI                    UPI
           UPI                    UPI
           UPI                    UPI
          Card                   Card
          Card                   Card
        Wallet                  Other
         Other                  Other


In [ ]:
# Our 150 students: how many distinct values does each column have?
print(students.nunique().to_string())
# One-hot on the student ID would add one column per training student, none of them useful for a new student
id_onehot = OneHotEncoder(handle_unknown="ignore").fit(students.loc[X_train.index, ["Student"]])
print("One-hot columns for Student on the training set:", len(id_onehot.categories_[0]))

Student          150
Class              3
Age                3
StudyHours        14
Attendance        42
PreviousScore     57
ExamScore         51
Passed             2
Source             2
One-hot columns for Student on the training set: 127


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Target-encoding leakage with an identifier. Naive: each student's "pass rate" computed on all training rows,
# which is their own result, then cross-validated. Safe: TargetEncoder inside the pipeline (out-of-fold on fit)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train = students.loc[X_train.index]
y = train["Passed"].astype(int)
numbers = ["StudyHours", "Attendance", "PreviousScore"]
lr = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
naive = train[numbers].assign(StudentRate=y.groupby(train["Student"]).transform("mean"))
safe = Pipeline([("prep", ColumnTransformer([("num", StandardScaler(), numbers),
                                             ("id", TargetEncoder(target_type="binary"), ["Student"])])),
                 ("model", LogisticRegression(max_iter=1000))])
print(f"Numbers only:                       CV accuracy {cross_val_score(lr, train[numbers], y, cv=cv).mean():.3f}")
print(f"+ student pass rate (naive, leaks): CV accuracy {cross_val_score(lr, naive, y, cv=cv).mean():.3f}")
print(f"+ TargetEncoder in the pipeline:    CV accuracy {cross_val_score(safe, train[numbers + ['Student']], y, cv=cv).mean():.3f}")

Numbers only:                       CV accuracy 0.796


+ student pass rate (naive, leaks): CV accuracy 1.000


+ TargetEncoder in the pipeline:    CV accuracy 0.796
